In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb
df = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/csv,xlsx/govno_expanded.xlsx')
df = df.drop(['lat_start','lon_start','lat_end','lon_end','id','city','is_indoor'],axis=1)
df = pd.get_dummies(df,columns=['category','type_of_movement'])
df['time(h)'] = pd.to_timedelta(df['time(h)']).dt.total_seconds()/3600

In [11]:
X = df.drop('name',axis=1)
pre_y = df['name']
le = LabelEncoder()
y = le.fit_transform(pre_y)
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.8, random_state=10)
clf = xgb.XGBRFClassifier(n_estimators=100, max_depth=8, random_state=10)
clf.fit(X_train,y_train)

XGBRFClassifier(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bytree=None, device=None,
                early_stopping_rounds=None, enable_categorical=False,
                eval_metric=None, feature_types=None, feature_weights=None,
                gamma=None, grow_policy=None, importance_type=None,
                interaction_constraints=None, max_bin=None,
                max_cat_threshold=None, max_cat_to_onehot=None,
                max_delta_step=None, max_depth=8, max_leaves=None,
                min_child_weight=None, missing=nan, monotone_constraints=None,
                multi_strategy=None, n_estimators=100, n_jobs=None,
                num_parallel_tree=None, objective='multi:softprob',
                random_state=10, ...)

In [14]:
import json
with open('data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
def transform_data(data):#time(h)
#time(h)
  if data['duration'] == "1_day":
    day = 24.0
  elif data['duration'] == "2_day":
    day = 48.0
  elif data['duration'] == "3_day":
    day = 72.0
  else:
    day = 120.0
  #with_child
  if data['company'] == "solo":
    cmp = 0
  elif data['company'] == "family":
    cmp=1
  else:
    cmp=0
  #type_of_movement
  if data['has_car'] == 'True':
      car='car'
  else:
    ind = np.random.randint(0,1)
    if ind==1:
      car = 'bus'
    else:
      car='pedestrian'
  #category
  dct=data['interests'].values
  if 'history' in dct:
    cat = 'historical'
  elif 'nature' in dct:
    cat = 'eco'
  elif 'food' in dct:
    cat = 'food'
  elif 'volunteer' in dct:
    cat = 'volunteer'
  #with_pets
  if data['with_pets'] == 'false':
    pets = 0
  else:
    pets = 1
  input_data = {
      'price(rub)': int(data['budget']),
      'time(h)': day,
      'type_of_movement': car,
      'category': cat,
      'with_child': cmp,
      'with_pets': pets
  }
  return input_data

FileNotFoundError: [Errno 2] No such file or directory: 'data.json'

In [15]:
def make_prediction(data):
    r_df = pd.DataFrame([transform_data[data]])
    for col in ['is_indoor', 'with_child', 'with_pets']:
        if col in r_df.columns:
            r_df[col] = r_df[col].astype(bool)

    cols_to_dummies = ['type_of_movement', 'category', 'city']
    r_df = pd.get_dummies(r_df, columns=cols_to_dummies, dtype=int)
    r_df = r_df.reindex(columns=X.columns, fill_value=0)
    predicted_probabilities = clf.predict_proba(r_df)[0]
    return predicted_probabilities

predicted_class_probabilities = make_prediction(input_data)
sorted_indices = np.argsort(predicted_class_probabilities)[::-1]

top_5_target_ids = sorted_indices[:5]
print(predicted_class_probabilities)

NameError: name 'transform_data' is not defined